# Comparison of simulation results with and without efficiency factor

The following results were obtained using the luminosity prescription from Pardo et al. (2025) based on a spectral index distribution with $\mu=-1.8$ and $\sigma=0.0$ and the corresponding best parameters. The following plots are based on a single realisation for the Survey Option 1 in Keane et al. (2025) and compare the impact for the efficiency factor based on incoherent pulsar searching introduced by Morello et al. (2020).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import rcParams
from matplotlib import rc
import matplotlib as mpl
import matplotlib.cm as cm
import matplotlib.colors as colors

rc("text", usetex=True)
rc("font", family="serif")
mpl.rcParams["text.latex.preamble"] = r"\usepackage{amsmath}"

In [ ]:
SMALL_SIZE = 30
MEDIUM_SIZE = 40
BIGGER_SIZE = 60

plt.rc("font", size=SMALL_SIZE)  # controls default text sizes
plt.rc("axes", titlesize=MEDIUM_SIZE)  # fontsize of the axes title
plt.rc("axes", labelsize=MEDIUM_SIZE)  # fontsize of the x and y labels
plt.rc("xtick", labelsize=MEDIUM_SIZE)  # fontsize of the tick labels
plt.rc("ytick", labelsize=MEDIUM_SIZE)  # fontsize of the tick labels
plt.rc("legend", fontsize=SMALL_SIZE)  # legend fontsize
plt.rc("figure", titlesize=BIGGER_SIZE)  # fontsize of the figure title

In [ ]:
colours = ["orange", "#FF5F15", "#0047AB", "purple", "black"]

## Loading the observed pulsar population

In [ ]:
# Read the full ATNF catalog.csv file. Binary pulsars are excluded.
df_atnf = pd.read_csv(
    "../../data/observations/atnf_full_nobinary_24-09-2024_with_errors.csv",
    delimiter=";",
    header=[0, 1],
)
df_atnf.head()

In [ ]:
# Select only those with measured period values.
df_atnf = df_atnf[~df_atnf["P0"]["(s)"].isin(["NAN"])]

# Remove those objects that are in globular clusters or in the Magellanic Clouds.
discard = [
    "EXGAL:SMC",
    "EXGAL:LMC",
    "GC:47Tuc",
    "GC:M3",
    "GC:M5",
    "GC:M13",
    "GC:NGC6440",
    "GC:Ter5",
    "GC:NGC6441",
    "GC:NGC6517",
    "GC:NGC6522",
    "GC:NGC6624",
    "GC:M28(NGC6626)",
    "GC:NGC6652",
    "GC:M22(NGC6656)",
    "GC:NGC6752",
    "GC:NGC6760",
    "GC:M15",
    "GC:M30",
]
df_atnf = df_atnf[
    ~df_atnf[("ASSOC", "Unnamed: 55_level_1")].str.match("|".join(discard))
]

len(df_atnf)

In [ ]:
df_atnf.head()

In [ ]:
df_atnf.columns = df_atnf.columns.droplevel(1)
df_atnf.head()

In [ ]:
df_atnf = df_atnf.drop(
    columns=[
        "#",
        "PX",
        "POSEPOCH",
        "DM",
        "TAU_SC",
        "S400",
        "S2000",
        "DIST",
        "XX",
        "YY",
    ],
)

len(df_atnf)

In [ ]:
# Select only isolated non-recycled neutron stars through filters with P > 0.01 and Pdot > 1e-19 (for those with measured values).
df_atnf = df_atnf[df_atnf["P0"].to_numpy().astype(np.float64) > 0.01]

len(df_atnf)

Note: For the purpose of modelling the observed isolated population of radio pulsars, in the following, we count those objects with period derivatives larger than $\dot{P} > 10^{-19} s/s$ or those with no measured period derivatives. The latter are likely isolated in nature due to the fact that only a small fraction of ATNF pulsars with known $\dot{P}$ actually have been recycled and attain $\dot{P} < 10^{-19} s/s$. As a result, the following number counts are slighlty larger than the populations used to produce our period period-derivative maps for the simulation-based inference approach.

In [ ]:
df_atnf = df_atnf[
    (df_atnf["P1"].to_numpy().astype(np.float64) > 1.0e-19)
    | (df_atnf["P1"].isin(["NAN"]))
]

In [ ]:
# Extract periods, period derivatives, longitude and latitude for all remaining pulsars.
P_obs = df_atnf["P0"].to_numpy().astype(np.float64)
Pdot_obs = df_atnf["P1"].to_numpy().astype(np.float64)
l_obs = df_atnf["Gl"].to_numpy().astype(np.float64)
b_obs = df_atnf["Gb"].to_numpy().astype(np.float64)

In [ ]:
l_obs[(l_obs > 180.0) & (l_obs < 360.0)] = (
    l_obs[(l_obs > 180.0) & (l_obs < 360.0)] - 360.0
)

In [ ]:
print(len(P_obs))

## Loading the simulated pulsar populations

In [ ]:
path_sim_1 = "../../SKA_census_simulations/survey_option1/run_1"

df_PMPS_sim_1 = pd.read_pickle(
    f"{path_sim_1}/survey_PMPS_results.pkl.gz",
    compression="gzip",
)

df_SMPS_sim_1 = pd.read_pickle(
    f"{path_sim_1}/survey_SMPS_results.pkl.gz",
    compression="gzip",
)

df_HTRU_sim_1 = pd.read_pickle(
    f"{path_sim_1}/survey_HTRU_low_mid_results.pkl.gz",
    compression="gzip",
)

df_SKA_low_sim_1 = pd.read_pickle(
    f"{path_sim_1}/survey_SKA_low_AA4_results.pkl.gz",
    compression="gzip",
)

df_SKA_mid1_sim_1 = pd.read_pickle(
    f"{path_sim_1}/survey_SKA_mid_band1_AA4_results.pkl.gz",
    compression="gzip",
)

df_SKA_mid2_sim_1 = pd.read_pickle(
    f"{path_sim_1}/survey_SKA_mid_band2_AA4_results.pkl.gz",
    compression="gzip",
)

path_sim_2 = (
    "../../SKA_census_simulations/survey_option1/test_efficiency_incoherent"
)

df_PMPS_sim_2 = pd.read_pickle(
    f"{path_sim_2}/survey_PMPS_results.pkl.gz",
    compression="gzip",
)

df_SMPS_sim_2 = pd.read_pickle(
    f"{path_sim_2}/survey_SMPS_results.pkl.gz",
    compression="gzip",
)

df_HTRU_sim_2 = pd.read_pickle(
    f"{path_sim_2}/survey_HTRU_low_mid_results.pkl.gz",
    compression="gzip",
)

df_SKA_low_sim_2 = pd.read_pickle(
    f"{path_sim_2}/survey_SKA_low_AA4_results.pkl.gz",
    compression="gzip",
)

df_SKA_mid1_sim_2 = pd.read_pickle(
    f"{path_sim_2}/survey_SKA_mid_band1_AA4_results.pkl.gz",
    compression="gzip",
)

df_SKA_mid2_sim_2 = pd.read_pickle(
    f"{path_sim_2}/survey_SKA_mid_band2_AA4_results.pkl.gz",
    compression="gzip",
)

## Counts for both simulation runs before death line is applied

Without efficiency factor.

In [ ]:
print("PMPS:", len(df_PMPS_sim_1))
print("SMPS:", len(df_SMPS_sim_1))
print("HTRU:", len(df_HTRU_sim_1))

In [ ]:
print("SKA low:", len(df_SKA_low_sim_1))
print("SKA mid1:", len(df_SKA_mid1_sim_1))
print("SKA mid2:", len(df_SKA_mid2_sim_1))

With efficiency factor.

In [ ]:
print("PMPS:", len(df_PMPS_sim_2))
print("SMPS:", len(df_SMPS_sim_2))
print("HTRU:", len(df_HTRU_sim_2))

In [ ]:
print("SKA low:", len(df_SKA_low_sim_2))
print("SKA mid1:", len(df_SKA_mid1_sim_2))
print("SKA mid2:", len(df_SKA_mid2_sim_2))

Reductions.

In [ ]:
print(
    "reduction PMPS:",
    100 * (len(df_PMPS_sim_1) - len(df_PMPS_sim_2)) / len(df_PMPS_sim_1),
)
print(
    "reduction SMPS:",
    100 * (len(df_SMPS_sim_1) - len(df_SMPS_sim_2)) / len(df_SMPS_sim_1),
)
print(
    "reduction HTRU:",
    100 * (len(df_HTRU_sim_1) - len(df_HTRU_sim_2)) / len(df_HTRU_sim_1),
)

In [ ]:
print(
    "reduction SKA low:",
    100
    * (len(df_SKA_low_sim_1) - len(df_SKA_low_sim_2))
    / len(df_SKA_low_sim_1),
)
print(
    "reduction SKA mid1:",
    100
    * (len(df_SKA_mid1_sim_1) - len(df_SKA_mid1_sim_2))
    / len(df_SKA_mid1_sim_1),
)
print(
    "reduction SKA mid2:",
    100
    * (len(df_SKA_mid2_sim_1) - len(df_SKA_mid2_sim_2))
    / len(df_SKA_mid2_sim_1),
)

## Loading P and Pdot values

In [ ]:
period_PMPS_sim_1 = df_PMPS_sim_1["P"]["[s]"].to_numpy()
period_SMPS_sim_1 = df_SMPS_sim_1["P"]["[s]"].to_numpy()
period_HTRU_sim_1 = df_HTRU_sim_1["P"]["[s]"].to_numpy()

pdot_PMPS_sim_1 = df_PMPS_sim_1["P_dot"]["[s s^-1]"].to_numpy()
pdot_SMPS_sim_1 = df_SMPS_sim_1["P_dot"]["[s s^-1]"].to_numpy()
pdot_HTRU_sim_1 = df_HTRU_sim_1["P_dot"]["[s s^-1]"].to_numpy()

In [ ]:
period_SKA_low_sim_1 = df_SKA_low_sim_1["P"]["[s]"].to_numpy()
period_SKA_mid1_sim_1 = df_SKA_mid1_sim_1["P"]["[s]"].to_numpy()
period_SKA_mid2_sim_1 = df_SKA_mid2_sim_1["P"]["[s]"].to_numpy()

pdot_SKA_low_sim_1 = df_SKA_low_sim_1["P_dot"]["[s s^-1]"].to_numpy()
pdot_SKA_mid1_sim_1 = df_SKA_mid1_sim_1["P_dot"]["[s s^-1]"].to_numpy()
pdot_SKA_mid2_sim_1 = df_SKA_mid2_sim_1["P_dot"]["[s s^-1]"].to_numpy()

In [ ]:
period_PMPS_sim_2 = df_PMPS_sim_2["P"]["[s]"].to_numpy()
period_SMPS_sim_2 = df_SMPS_sim_2["P"]["[s]"].to_numpy()
period_HTRU_sim_2 = df_HTRU_sim_2["P"]["[s]"].to_numpy()

pdot_PMPS_sim_2 = df_PMPS_sim_2["P_dot"]["[s s^-1]"].to_numpy()
pdot_SMPS_sim_2 = df_SMPS_sim_2["P_dot"]["[s s^-1]"].to_numpy()
pdot_HTRU_sim_2 = df_HTRU_sim_2["P_dot"]["[s s^-1]"].to_numpy()

In [ ]:
period_SKA_low_sim_2 = df_SKA_low_sim_2["P"]["[s]"].to_numpy()
period_SKA_mid1_sim_2 = df_SKA_mid1_sim_2["P"]["[s]"].to_numpy()
period_SKA_mid2_sim_2 = df_SKA_mid2_sim_2["P"]["[s]"].to_numpy()

pdot_SKA_low_sim_2 = df_SKA_low_sim_2["P_dot"]["[s s^-1]"].to_numpy()
pdot_SKA_mid1_sim_2 = df_SKA_mid1_sim_2["P_dot"]["[s s^-1]"].to_numpy()
pdot_SKA_mid2_sim_2 = df_SKA_mid2_sim_2["P_dot"]["[s s^-1]"].to_numpy()

## Counts after accounting for the death line

In [ ]:
b = 10

Without efficiency factor.

In [ ]:
dl_PMPS_sim_1 = (
    2.1e-18 * (1.4 / 1) ** (-1) * b ** (-0.5) * (period_PMPS_sim_1) ** 2
)
dl_SMPS_sim_1 = (
    2.1e-18 * (1.4 / 1) ** (-1) * b ** (-0.5) * (period_SMPS_sim_1) ** 2
)
dl_HTRU_sim_1 = (
    2.1e-18 * (1.4 / 1) ** (-1) * b ** (-0.5) * (period_HTRU_sim_1) ** 2
)

above_dl_PMPS_sim_1 = sum(pdot_PMPS_sim_1 > dl_PMPS_sim_1)
above_dl_SMPS_sim_1 = sum(pdot_SMPS_sim_1 > dl_SMPS_sim_1)
above_dl_HTRU_sim_1 = sum(pdot_HTRU_sim_1 > dl_HTRU_sim_1)

In [ ]:
print("PMPS:", above_dl_PMPS_sim_1)
print("SMPS:", above_dl_SMPS_sim_1)
print("HTRU:", above_dl_HTRU_sim_1)

In [ ]:
dl_SKA_low_sim_1 = (
    2.1e-18 * (1.4 / 1) ** (-1) * b ** (-0.5) * (period_SKA_low_sim_1) ** 2
)
dl_SKA_mid1_sim_1 = (
    2.1e-18 * (1.4 / 1) ** (-1) * b ** (-0.5) * (period_SKA_mid1_sim_1) ** 2
)
dl_SKA_mid2_sim_1 = (
    2.1e-18 * (1.4 / 1) ** (-1) * b ** (-0.5) * (period_SKA_mid2_sim_1) ** 2
)

above_dl_SKA_low_sim_1 = sum(pdot_SKA_low_sim_1 > dl_SKA_low_sim_1)
above_dl_SKA_mid1_sim_1 = sum(pdot_SKA_mid1_sim_1 > dl_SKA_mid1_sim_1)
above_dl_SKA_mid2_sim_1 = sum(pdot_SKA_mid2_sim_1 > dl_SKA_mid2_sim_1)

In [ ]:
print("SKA low:", above_dl_SKA_low_sim_1)
print("SKA mid1:", above_dl_SKA_mid1_sim_1)
print("SKA mid2:", above_dl_SKA_mid2_sim_1)

In [ ]:
dl_PMPS_sim_2 = (
    2.1e-18 * (1.4 / 1) ** (-1) * b ** (-0.5) * (period_PMPS_sim_2) ** 2
)
dl_SMPS_sim_2 = (
    2.1e-18 * (1.4 / 1) ** (-1) * b ** (-0.5) * (period_SMPS_sim_2) ** 2
)
dl_HTRU_sim_2 = (
    2.1e-18 * (1.4 / 1) ** (-1) * b ** (-0.5) * (period_HTRU_sim_2) ** 2
)

above_dl_PMPS_sim_2 = sum(pdot_PMPS_sim_2 > dl_PMPS_sim_2)
above_dl_SMPS_sim_2 = sum(pdot_SMPS_sim_2 > dl_SMPS_sim_2)
above_dl_HTRU_sim_2 = sum(pdot_HTRU_sim_2 > dl_HTRU_sim_2)

In [ ]:
print("PMPS:", above_dl_PMPS_sim_2)
print("SMPS:", above_dl_SMPS_sim_2)
print("HTRU:", above_dl_HTRU_sim_2)

In [ ]:
dl_SKA_low_sim_2 = (
    2.1e-18 * (1.4 / 1) ** (-1) * b ** (-0.5) * (period_SKA_low_sim_2) ** 2
)
dl_SKA_mid1_sim_2 = (
    2.1e-18 * (1.4 / 1) ** (-1) * b ** (-0.5) * (period_SKA_mid1_sim_2) ** 2
)
dl_SKA_mid2_sim_2 = (
    2.1e-18 * (1.4 / 1) ** (-1) * b ** (-0.5) * (period_SKA_mid2_sim_2) ** 2
)

above_dl_SKA_low_sim_2 = sum(pdot_SKA_low_sim_2 > dl_SKA_low_sim_2)
above_dl_SKA_mid1_sim_2 = sum(pdot_SKA_mid1_sim_2 > dl_SKA_mid1_sim_2)
above_dl_SKA_mid2_sim_2 = sum(pdot_SKA_mid2_sim_2 > dl_SKA_mid2_sim_2)

In [ ]:
print("SKA low:", above_dl_SKA_low_sim_2)
print("SKA mid1:", above_dl_SKA_mid1_sim_2)
print("SKA mid2:", above_dl_SKA_mid2_sim_2)

Reductions.

In [ ]:
print(
    "reduction PMPS:",
    100 * (above_dl_PMPS_sim_1 - above_dl_PMPS_sim_2) / above_dl_PMPS_sim_1,
)
print(
    "reduction SMPS:",
    100 * (above_dl_PMPS_sim_1 - above_dl_PMPS_sim_2) / above_dl_PMPS_sim_1,
)
print(
    "reduction HTRU:",
    100 * (above_dl_HTRU_sim_1 - above_dl_HTRU_sim_2) / above_dl_HTRU_sim_1,
)

In [ ]:
print(
    "reduction SKA low:",
    100
    * (above_dl_SKA_low_sim_1 - above_dl_SKA_low_sim_2)
    / above_dl_SKA_low_sim_1,
)
print(
    "reduction SKA mid1:",
    100
    * (above_dl_SKA_mid1_sim_1 - above_dl_SKA_mid1_sim_2)
    / above_dl_SKA_mid1_sim_1,
)
print(
    "reduction SKA mid2:",
    100
    * (above_dl_SKA_mid2_sim_1 - above_dl_SKA_mid2_sim_2)
    / above_dl_SKA_mid2_sim_1,
)

## Plotting P and Pdot histograms.

In [ ]:
bins_period = np.linspace(-2, 2, 50)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))

ax.hist(
    np.log10(P_obs),
    bins=bins_period,
    histtype="step",
    color="black",
    lw=4,
    alpha=0.8,
    label=r"Observed full",
    density=True,
)
ax.hist(
    np.log10(period_PMPS_sim_1),
    bins=bins_period,
    histtype="step",
    color="blue",
    lw=4,
    alpha=0.3,
    label=r"PMPS without $\epsilon$",
    density=True,
)
ax.hist(
    np.log10(period_PMPS_sim_2),
    bins=bins_period,
    histtype="step",
    color="blue",
    linestyle="--",
    lw=4,
    alpha=0.8,
    label=r"PMPS with $\epsilon$",
    density=True,
)
ax.hist(
    np.log10(period_SMPS_sim_1),
    bins=bins_period,
    histtype="step",
    color="orange",
    lw=4,
    alpha=0.3,
    label=r"SMPS without $\epsilon$",
    density=True,
)
ax.hist(
    np.log10(period_SMPS_sim_2),
    bins=bins_period,
    histtype="step",
    color="orange",
    linestyle="--",
    lw=4,
    alpha=0.8,
    label=r"SMPS with $\epsilon$",
    density=True,
)
ax.hist(
    np.log10(period_HTRU_sim_1),
    bins=bins_period,
    histtype="step",
    color="red",
    lw=4,
    alpha=0.3,
    label=r"HTRU without $\epsilon$",
    density=True,
)
ax.hist(
    np.log10(period_HTRU_sim_2),
    bins=bins_period,
    histtype="step",
    color="red",
    linestyle="--",
    lw=4,
    alpha=0.8,
    label=r"HTRU with $\epsilon$",
    density=True,
)

plt.xlabel(r"Period [s]")
plt.ylabel(r"PDF of detected stars")
ax.set_xlim(-2.0, 2.5)
ax.set_ylim(0.0, 1.7)
plt.legend(frameon=True, loc=1)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))

ax.hist(
    np.log10(P_obs),
    bins=bins_period,
    histtype="step",
    color="black",
    lw=4,
    alpha=0.8,
    label=r"Observed full",
    density=True,
)
ax.hist(
    np.log10(period_SKA_low_sim_1),
    bins=bins_period,
    histtype="step",
    color="blue",
    lw=4,
    alpha=0.3,
    label=r"SKA low without $\epsilon$",
    density=True,
)
ax.hist(
    np.log10(period_SKA_low_sim_2),
    bins=bins_period,
    histtype="step",
    color="blue",
    linestyle="--",
    lw=4,
    alpha=0.8,
    label=r"SKA low with $\epsilon$",
    density=True,
)
ax.hist(
    np.log10(period_SKA_mid1_sim_1),
    bins=bins_period,
    histtype="step",
    color="orange",
    lw=4,
    alpha=0.3,
    label=r"SKA mid1 without $\epsilon$",
    density=True,
)
ax.hist(
    np.log10(period_SKA_mid1_sim_2),
    bins=bins_period,
    histtype="step",
    color="orange",
    linestyle="--",
    lw=4,
    alpha=0.8,
    label=r"SKA mid1 with $\epsilon$",
    density=True,
)
ax.hist(
    np.log10(period_SKA_mid2_sim_1),
    bins=bins_period,
    histtype="step",
    color="red",
    lw=4,
    alpha=0.3,
    label=r"SKA mid2 without $\epsilon$",
    density=True,
)
ax.hist(
    np.log10(period_SKA_mid2_sim_2),
    bins=bins_period,
    histtype="step",
    color="red",
    linestyle="--",
    lw=4,
    alpha=0.8,
    label=r"SKA mid2 with $\epsilon$",
    density=True,
)

plt.xlabel(r"Period [s]")
plt.ylabel(r"PDF of detected stars")
ax.set_xlim(-2.0, 2.5)
ax.set_ylim(0.0, 1.7)
plt.legend(frameon=True, loc=1)

plt.show()

In [ ]:
bins_pdot = np.linspace(-22, -10, 50)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))

ax.hist(
    np.log10(Pdot_obs),
    bins=bins_pdot,
    histtype="step",
    color="black",
    lw=4,
    alpha=0.8,
    label=r"Observed",
    density=True,
)
ax.hist(
    np.log10(pdot_PMPS_sim_1),
    bins=bins_pdot,
    histtype="step",
    color="blue",
    lw=4,
    alpha=0.3,
    label=r"PMPS without $\epsilon$",
    density=True,
)
ax.hist(
    np.log10(pdot_PMPS_sim_2),
    bins=bins_pdot,
    histtype="step",
    color="blue",
    linestyle="--",
    lw=4,
    alpha=0.8,
    label=r"PMPS with $\epsilon$",
    density=True,
)
ax.hist(
    np.log10(pdot_SMPS_sim_1),
    bins=bins_pdot,
    histtype="step",
    color="orange",
    lw=4,
    alpha=0.3,
    label=r"SMPS without $\epsilon$",
    density=True,
)
ax.hist(
    np.log10(pdot_SMPS_sim_2),
    bins=bins_pdot,
    histtype="step",
    color="orange",
    linestyle="--",
    lw=4,
    alpha=0.8,
    label=r"SMPS with $\epsilon$",
    density=True,
)
ax.hist(
    np.log10(pdot_HTRU_sim_1),
    bins=bins_pdot,
    histtype="step",
    color="red",
    lw=4,
    alpha=0.3,
    label=r"HTRU without $\epsilon$",
    density=True,
)
ax.hist(
    np.log10(pdot_HTRU_sim_2),
    bins=bins_pdot,
    histtype="step",
    color="red",
    linestyle="--",
    lw=4,
    alpha=0.8,
    label=r"HTRU with $\epsilon$",
    density=True,
)

plt.xlabel(r"Period derivative [s s$^{-1}$]")
plt.ylabel(r"PDF of detected stars")
ax.set_xlim(-20.0, -8.0)
ax.set_ylim(0.0, 0.8)
plt.legend(frameon=True, loc=1)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))

ax.hist(
    np.log10(Pdot_obs),
    bins=bins_pdot,
    histtype="step",
    color="black",
    lw=4,
    alpha=0.8,
    label=r"Observed full",
    density=True,
)
ax.hist(
    np.log10(pdot_SKA_low_sim_1),
    bins=bins_pdot,
    histtype="step",
    color="blue",
    lw=4,
    alpha=0.3,
    label=r"SKA low without $\epsilon$",
    density=True,
)
ax.hist(
    np.log10(pdot_SKA_low_sim_2),
    bins=bins_pdot,
    histtype="step",
    color="blue",
    linestyle="--",
    lw=4,
    alpha=0.8,
    label=r"SKA low with $\epsilon$",
    density=True,
)
ax.hist(
    np.log10(pdot_SKA_mid1_sim_1),
    bins=bins_pdot,
    histtype="step",
    color="orange",
    lw=4,
    alpha=0.3,
    label=r"SKA mid1 without $\epsilon$",
    density=True,
)
ax.hist(
    np.log10(pdot_SKA_mid1_sim_2),
    bins=bins_pdot,
    histtype="step",
    color="orange",
    linestyle="--",
    lw=4,
    alpha=0.8,
    label=r"SKA mid1 with $\epsilon$",
    density=True,
)
ax.hist(
    np.log10(pdot_SKA_mid2_sim_1),
    bins=bins_pdot,
    histtype="step",
    color="red",
    lw=4,
    alpha=0.3,
    label=r"SKA mid2 without $\epsilon$",
    density=True,
)
ax.hist(
    np.log10(pdot_SKA_mid2_sim_2),
    bins=bins_pdot,
    histtype="step",
    color="red",
    linestyle="--",
    lw=4,
    alpha=0.8,
    label=r"SKA mid2 with $\epsilon$",
    density=True,
)

plt.xlabel(r"Period derivative [s s$^{-1}$]")
plt.ylabel(r"PDF of detected stars")
ax.set_xlim(-22.0, -8.0)
ax.set_ylim(0.0, 0.8)
plt.legend(frameon=True, loc=1)

plt.show()

## Pdot histrograms for sources above the death line

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))

ax.hist(
    np.log10(Pdot_obs),
    bins=bins_pdot,
    histtype="step",
    color="black",
    lw=4,
    alpha=0.8,
    label=r"Observed",
    density=True,
)
ax.hist(
    np.log10(pdot_PMPS_sim_1[pdot_PMPS_sim_1 > dl_PMPS_sim_1]),
    bins=bins_pdot,
    histtype="step",
    color="blue",
    lw=4,
    alpha=0.3,
    label=r"PMPS without $\epsilon$",
    density=True,
)
ax.hist(
    np.log10(pdot_PMPS_sim_2[pdot_PMPS_sim_2 > dl_PMPS_sim_2]),
    bins=bins_pdot,
    histtype="step",
    color="blue",
    linestyle="--",
    lw=4,
    alpha=0.3,
    label=r"PMPS without $\epsilon$",
    density=True,
)
ax.hist(
    np.log10(pdot_SMPS_sim_1[pdot_SMPS_sim_1 > dl_SMPS_sim_1]),
    bins=bins_pdot,
    histtype="step",
    color="orange",
    lw=4,
    alpha=0.3,
    label=r"SMPS without $\epsilon$",
    density=True,
)
ax.hist(
    np.log10(pdot_SMPS_sim_2[pdot_SMPS_sim_2 > dl_SMPS_sim_2]),
    bins=bins_pdot,
    histtype="step",
    color="orange",
    linestyle="--",
    lw=4,
    alpha=0.8,
    label=r"SMPS with $\epsilon$",
    density=True,
)
ax.hist(
    np.log10(pdot_HTRU_sim_1[pdot_HTRU_sim_1 > dl_HTRU_sim_1]),
    bins=bins_pdot,
    histtype="step",
    color="red",
    lw=4,
    alpha=0.3,
    label=r"HTRU without $\epsilon$",
    density=True,
)
ax.hist(
    np.log10(pdot_HTRU_sim_2[pdot_HTRU_sim_2 > dl_HTRU_sim_2]),
    bins=bins_pdot,
    histtype="step",
    color="red",
    linestyle="--",
    lw=4,
    alpha=0.8,
    label=r"HTRU with $\epsilon$",
    density=True,
)

plt.xlabel(r"Period derivative [s s$^{-1}$]")
plt.ylabel(r"PDF of detected stars")
ax.set_xlim(-20.0, -8.0)
ax.set_ylim(0.0, 0.8)
plt.legend(frameon=True, loc=1)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))

ax.hist(
    np.log10(Pdot_obs),
    bins=bins_pdot,
    histtype="step",
    color="black",
    lw=4,
    alpha=0.8,
    label=r"Observed",
    density=True,
)
ax.hist(
    np.log10(pdot_SKA_low_sim_1[pdot_SKA_low_sim_1 > dl_SKA_low_sim_1]),
    bins=bins_pdot,
    histtype="step",
    color="blue",
    lw=4,
    alpha=0.3,
    label=r"SKA low without $\epsilon$",
    density=True,
)
ax.hist(
    np.log10(pdot_SKA_low_sim_2[pdot_SKA_low_sim_2 > dl_SKA_low_sim_2]),
    bins=bins_pdot,
    histtype="step",
    color="blue",
    linestyle="--",
    lw=4,
    alpha=0.3,
    label=r"SKA low without $\epsilon$",
    density=True,
)
ax.hist(
    np.log10(pdot_SKA_mid1_sim_1[pdot_SKA_mid1_sim_1 > dl_SKA_mid1_sim_1]),
    bins=bins_pdot,
    histtype="step",
    color="orange",
    lw=4,
    alpha=0.3,
    label=r"SKA mid1 without $\epsilon$",
    density=True,
)
ax.hist(
    np.log10(pdot_SKA_mid1_sim_2[pdot_SKA_mid1_sim_2 > dl_SKA_mid1_sim_2]),
    bins=bins_pdot,
    histtype="step",
    color="orange",
    linestyle="--",
    lw=4,
    alpha=0.8,
    label=r"SKA mid1 with $\epsilon$",
    density=True,
)
ax.hist(
    np.log10(pdot_SKA_mid2_sim_1[pdot_SKA_mid2_sim_1 > dl_SKA_mid2_sim_1]),
    bins=bins_pdot,
    histtype="step",
    color="red",
    lw=4,
    alpha=0.3,
    label=r"SKA mid2 without $\epsilon$",
    density=True,
)
ax.hist(
    np.log10(pdot_SKA_mid2_sim_2[pdot_SKA_mid2_sim_2 > dl_SKA_mid2_sim_2]),
    bins=bins_pdot,
    histtype="step",
    color="red",
    linestyle="--",
    lw=4,
    alpha=0.8,
    label=r"SKA mid2 with $\epsilon$",
    density=True,
)

plt.xlabel(r"Period derivative [s s$^{-1}$]")
plt.ylabel(r"PDF of detected stars")
ax.set_xlim(-22.0, -8.0)
ax.set_ylim(0.0, 0.8)
plt.legend(frameon=True, loc=1)

plt.show()

Conclusion: Distributions don't change only the birth rate, so a new inference experiment should not really affect the best-fit parameters.